# Scientific Interest in AMR

This notebook is the yearly version of the scientific-interest workflow. It uses `pmids_results_positive_yearly.parquet`, so each observation is now a true `(pathogen, antibiotic, year)` count rather than a 5-year interval count.

The main idea stays the same:

- `count` is the observed AMR-positive publication count for one `(pathogen, antibiotic, year)` row
- `bg_pathogen_n` is the total publication background for that pathogen in the same year
- `bg_antibiotic_n` is the total publication background for that antibiotic in the same year
- the pooled model defines the expected AMR count from general pathogen and antibiotic attention
- `SI` is the residual gap between the observed log-count and that pooled expectation

Because the data are now yearly, the notebook can work directly on annual points without fitting over 5-year interval end-years.


In [ ]:
%config InlineBackend.figure_format = 'retina'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from IPython.display import display


## Load yearly AMR counts and yearly backgrounds

This notebook expects the yearly positive-AMR parquet created in `6_yearly_pubmed_dates.ipynb`. The pathogen and antibiotic background tables were already collected at yearly resolution, so we can merge them directly on `year`.


In [ ]:
YEARLY_PATH = "pmids_results_positive_yearly.parquet"
PATHOGEN_BG_PATH = "pathogen_tiab_counts.parquet"
ANTIBIOTIC_BG_PATH = "antibiotic_tiab_counts.parquet"

yearly_df = pd.read_parquet(YEARLY_PATH)
pathogen_bg_df = pd.read_parquet(PATHOGEN_BG_PATH).rename(columns={"count": "bg_pathogen_n"})
antibiotic_bg_df = pd.read_parquet(ANTIBIOTIC_BG_PATH).rename(columns={"count": "bg_antibiotic_n"})

amr_yearly_df = (
    yearly_df.merge(pathogen_bg_df, on=["pathogen", "year"], how="left")
    .merge(antibiotic_bg_df, on=["antibiotic", "year"], how="left")
    .copy()
)

amr_yearly_df["count"] = pd.to_numeric(amr_yearly_df["count"], errors="coerce")
amr_yearly_df["bg_pathogen_n"] = pd.to_numeric(amr_yearly_df["bg_pathogen_n"], errors="coerce")
amr_yearly_df["bg_antibiotic_n"] = pd.to_numeric(amr_yearly_df["bg_antibiotic_n"], errors="coerce")
amr_yearly_df["log_count"] = np.log1p(amr_yearly_df["count"])
amr_yearly_df["log_bg_pathogen_n"] = np.log1p(amr_yearly_df["bg_pathogen_n"])
amr_yearly_df["log_bg_antibiotic_n"] = np.log1p(amr_yearly_df["bg_antibiotic_n"])

amr_yearly_df = amr_yearly_df.dropna(
    subset=["count", "bg_pathogen_n", "bg_antibiotic_n", "log_count", "log_bg_pathogen_n", "log_bg_antibiotic_n"]
).reset_index(drop=True)

print(f"yearly AMR rows: {len(amr_yearly_df):,}")
print(f"unique pathogen-antibiotic pairs: {amr_yearly_df[['pathogen', 'antibiotic']].drop_duplicates().shape[0]:,}")
print(f"year span: {int(amr_yearly_df['year'].min())} to {int(amr_yearly_df['year'].max())}")

display(amr_yearly_df.head())


## Pooled yearly baseline model

We fit the same pooled baseline as before, but now at yearly resolution:

`log1p(count)` = `alpha` + `u_pathogen` + `v_antibiotic` + `beta_path * log1p(bg_pathogen_n)` + `beta_ab * log1p(bg_antibiotic_n)`

Here:

- `alpha` is the global intercept
- `u_pathogen` is the pathogen-specific pooled effect
- `v_antibiotic` is the antibiotic-specific pooled effect
- `beta_path` and `beta_ab` tell us how expected AMR counts scale with yearly pathogen and antibiotic attention

Once the model is fitted, the yearly scientific-interest score is:

`SI = log1p(count) - expected_log_count_global`


In [ ]:
alpha = 1.0

pathogen_dummies = pd.get_dummies(amr_yearly_df["pathogen"].astype(str), prefix="pathogen", dtype=float)
antibiotic_dummies = pd.get_dummies(amr_yearly_df["antibiotic"].astype(str), prefix="antibiotic", dtype=float)

design_df = pd.concat(
    [
        amr_yearly_df[["log_bg_pathogen_n", "log_bg_antibiotic_n"]].reset_index(drop=True),
        pathogen_dummies.reset_index(drop=True),
        antibiotic_dummies.reset_index(drop=True),
    ],
    axis=1,
)

response = amr_yearly_df["log_count"].to_numpy(dtype=float)
pooled_model = Ridge(alpha=alpha, fit_intercept=True)
pooled_model.fit(design_df, response)

si_df = amr_yearly_df.copy()
si_df["expected_log_count_global"] = pooled_model.predict(design_df)
si_df["expected_count_global"] = np.clip(np.expm1(si_df["expected_log_count_global"]), 0.0, None)
si_df["SI"] = si_df["log_count"] - si_df["expected_log_count_global"]

display(
    si_df[
        [
            "pathogen",
            "antibiotic",
            "year",
            "count",
            "bg_pathogen_n",
            "bg_antibiotic_n",
            "expected_count_global",
            "SI",
        ]
    ].head()
)


## Inspect the fitted baseline

The two continuous coefficients are usually the most interpretable summary of the pooled model, because they show how expected AMR publication volume scales with general yearly attention to the pathogen and the antibiotic.


In [ ]:
coef_df = pd.DataFrame({"feature": design_df.columns, "coefficient": pooled_model.coef_})
display(coef_df[coef_df["feature"].isin(["log_bg_pathogen_n", "log_bg_antibiotic_n"])])
print("Model intercept:", float(pooled_model.intercept_))


## Rank pairs by average yearly scientific interest

This gives a compact overview of which pathogen-antibiotic pairs are persistently above the pooled yearly baseline.


In [ ]:
pair_summary_df = (
    si_df.groupby(["pathogen", "antibiotic"], as_index=False)
    .agg(
        mean_SI=("SI", "mean"),
        total_count=("count", "sum"),
        mean_count=("count", "mean"),
        n_years=("year", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .sort_values(["mean_SI", "total_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(pair_summary_df.head(20))


## Plot one yearly pair trajectory

With yearly data, we can plot the observed count trajectory and the yearly `SI` directly. The optional rolling mean just makes the annual pattern easier to read; it is not part of the SI definition.


In [ ]:
def plot_yearly_si_pair(
    si_df: pd.DataFrame,
    *,
    pathogen: str,
    antibiotic: str,
    rolling_window: int = 3,
    figsize: tuple[int, int] = (12, 7),
):
    pair_df = (
        si_df.loc[
            (si_df["pathogen"] == pathogen) & (si_df["antibiotic"] == antibiotic),
            [
                "year",
                "count",
                "expected_count_global",
                "SI",
                "bg_pathogen_n",
                "bg_antibiotic_n",
            ],
        ]
        .sort_values("year")
        .reset_index(drop=True)
    )
    if pair_df.empty:
        raise ValueError(f"No yearly SI rows found for pathogen={pathogen!r}, antibiotic={antibiotic!r}.")

    pair_df["count_roll"] = pair_df["count"].rolling(window=rolling_window, center=True, min_periods=1).mean()
    pair_df["si_roll"] = pair_df["SI"].rolling(window=rolling_window, center=True, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=figsize, facecolor="white")
    ax2 = ax.twinx()
    ax.set_facecolor("#fbfbf8")
    ax2.set_facecolor("none")

    ax2.bar(
        pair_df["year"],
        np.clip(pair_df["count"], 0.0, None),
        width=0.85,
        color="#9ecae1",
        alpha=0.50,
        edgecolor="none",
        label="Observed AMR count",
        zorder=1,
    )
    ax2.plot(
        pair_df["year"],
        pair_df["expected_count_global"],
        color="#4e8fb3",
        linewidth=2.4,
        label="Expected count (pooled baseline)",
        zorder=2,
    )
    ax2.plot(
        pair_df["year"],
        pair_df["count_roll"],
        color="#1f5a7a",
        linewidth=2.8,
        linestyle="--",
        label=f"Observed count ({rolling_window}-year rolling mean)",
        zorder=3,
    )

    ax.axhline(0.0, color="#6b7280", linewidth=1.4, linestyle="--", alpha=0.9)
    ax.plot(
        pair_df["year"],
        pair_df["SI"],
        color="#111111",
        marker="o",
        linewidth=2.2,
        markersize=4.8,
        label="Observed yearly SI",
        zorder=5,
    )
    ax.plot(
        pair_df["year"],
        pair_df["si_roll"],
        color="#d62828",
        linewidth=2.8,
        label=f"SI ({rolling_window}-year rolling mean)",
        zorder=6,
    )

    ax.set_xlabel("Year")
    ax.set_ylabel("Scientific Interest (SI)")
    ax2.set_ylabel("AMR publication counts")
    ax.set_title(f"Yearly Scientific Interest: {pathogen} vs {antibiotic}", fontsize=15, pad=10)
    ax.grid(True, axis="y", color="#c7c7c7", alpha=0.45, linewidth=0.8)
    ax.grid(False, axis="x")

    handles1, labels1 = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(
        handles1 + handles2,
        labels1 + labels2,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.14),
        ncol=2,
        frameon=False,
    )
    plt.tight_layout(rect=(0, 0.06, 1, 1))
    plt.show()
    return pair_df


In [ ]:
example_pair_df = plot_yearly_si_pair(
    si_df,
    pathogen="Enterococcus faecium",
    antibiotic="Vancomycin",
    rolling_window=3,
)
display(example_pair_df.tail(15))
